In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

print('PyTorch version:', torch.__version__)

PyTorch version: 2.12.0.dev20260408+cu128


In [4]:
#dropout的torch实现：手写与torch自带的dropout对比
#手写dropout
def dropout(x: Tensor, #激活值x
            p:float=0.5,
            training:bool=True) -> Tensor:
    if not 0.0<=p<=1.0:
        raise ValueError(f'p must be between 0 and 1, but got {p}')
    if not training or p==0.0:
        return x
    if p==1.0:
        return torch.zeros_like(x)

    keep=1.0-p
    mask=torch.rand_like(x)<keep #torch.rand_like(x)生成一个和激活值 x 形状一样的张量，里面的值是 [0, 1) 之间均匀分布的随机浮点数。< keep若该张量里面元素小于keep保留概率，则变为1。mask就此形成
    return x*mask/keep

#将dropout打包成nn.Module类
class Dropout(nn.Module):
    def __init__(self,p:float=0.5):
        super().__init__()
        if not 0.0<=p<=1.0:
            raise ValueError(f'p must be between 0 and 1, but got {p}')
        self.p=p

    def forward(self,x: Tensor) -> Tensor:
        return dropout(x,p=self.p,training=self.training)

    def extra_repr(self)->str:
        return f'p={self.p}'

#手写与torch自带的dropout对比
dropout1=Dropout() #实例化Dropout为dropout1
x=torch.randn(10,dtype=torch.float32)

actual=dropout1(x) #隐藏某个激活值的概率为0.25
expected=F.dropout(x,p=0.5)

print('Custom Dropout:', actual)
print('Custom Dropout:', expected)

Custom Dropout: tensor([ 0.7911, -0.0000, -1.2897,  0.0000,  3.1655,  0.5374, -0.0000, -0.0999,
        -2.1223, -1.0958])
Custom Dropout: tensor([ 0.7911, -1.2563, -0.0000,  2.8722,  0.0000,  0.5374, -0.1551, -0.0000,
        -0.0000, -1.0958])


In [4]:
#开启/关闭训练模式&推理模式
dropout=nn.Dropout(p=0.5)
x=torch.ones(10) #激活特征

#训练阶段：关闭部分激活特征
dropout.train()
train_output1=dropout(x)
train_output2=dropout(x)

#推理阶段：开启所有激活特征
dropout.eval()
eval_output1=dropout(x)
eval_output2=dropout(x)

print('Training output1:',train_output1)
print('Training output2:',train_output2)
print('Evaluation output1:',eval_output1)
print('Evaluation output2:',eval_output2)

Training output1: tensor([0., 0., 0., 0., 2., 0., 0., 2., 2., 0.])
Training output2: tensor([2., 0., 0., 0., 2., 2., 2., 0., 2., 0.])
Evaluation output1: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
Evaluation output2: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


In [9]:
#dropout反向传播
x=torch.randn(12,requires_grad=True)
y=dropout(x,p=0.5)

y.backward(gradient=torch.ones_like(y)) #沿着计算图反向传播。torch.ones_like(y)仿照y造一个全1张量，并通过 gradient=ones_like(y) 隐式定义成了 loss = y.sum()变成标量

print('Dropout outout:',y.data) #y.data：返回与 y 共享内存但脱离计算图的张量
print('Input gradient:',x.grad) #x.grad就是再算loss对x的梯度

Dropout outout: tensor([ 0.0000, -0.0000,  2.3252, -2.3521,  0.0000,  0.0000,  0.0000, -2.8915,
         0.0000, -0.0000,  1.7974, -3.5691])
Input gradient: tensor([0., 0., 2., 2., 0., 0., 0., 2., 0., 0., 2., 2.])
